# Notebook 15 — Neural 3D Reconstruction

**Vision & 3D Mapping Workshop** | Block 4: SLAM & Neural 3D

---

## Why This Matters

Neural 3D reconstruction has revolutionized how we represent and render 3D scenes.
Starting with NeRF (2020), the field has moved from implicit neural representations to
explicit 3D Gaussian Splatting (2023), enabling real-time rendering at unprecedented
quality. These techniques are now being integrated into SLAM systems, generative models,
and industrial applications.

This notebook derives the volume rendering equation from first principles, implements ray
marching, builds a full 3D Gaussian Splatting forward model from scratch, and explores
the rapidly evolving landscape of neural 3D methods.

### What You'll Learn

1. **Volume rendering equation** — continuous and discrete forms, with implementation
2. **NeRF architecture** — positional encoding, hierarchical sampling
3. **3D Gaussian Splatting** — full forward model from scratch
4. **Differentiable rendering** — why gradients flow through 3DGS
5. **3DGS-SLAM** — SplaTAM, MonoGS, EGG-Fusion
6. **MASt3R-SLAM** — foundation models for SLAM
7. **Neural implicit surfaces** — NeuS, VolSDF
8. **Inverse rendering** — material decomposition
9. **Non-rigid SLAM** — deformable scenes
10. **Generative 3D** — DreamFusion, GSGEN, PixGS

### Prerequisites
- Notebook 03 (Camera Models)
- Notebook 04 (Rotations and Poses)
- Notebook 14 (Visual SLAM)

### References
- Mildenhall et al., "NeRF: Representing Scenes as Neural Radiance Fields", ECCV 2020
- Kerbl et al., "3D Gaussian Splatting for Real-Time Radiance Field Rendering", SIGGRAPH 2023
- Zwicker et al., "EWA Splatting", IEEE TVCG, 2002
- Max, "Optical Models for Direct Volume Rendering", IEEE TVCG, 1995

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
from numpy.linalg import inv, norm, det
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.spatial.transform import Rotation
import warnings
warnings.filterwarnings('ignore')

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'font.size': 11, 'figure.figsize': (10, 6)})



## 0. Image Formation & Radiometry

*Before* understanding how NeRF renders images, we need to understand
how light becomes pixels in the first place.

### Radiometric Quantities

| Quantity | Symbol | Units | Meaning |
|----------|--------|-------|---------|
| **Flux** | $\Phi$ | W | Total power of light |
| **Irradiance** | $E$ | W/m² | Power per unit area (arriving at surface) |
| **Radiance** | $L$ | W/(m²·sr) | Power per unit area per solid angle |

**Key property**: radiance is **constant along rays** through empty space
(no absorption/emission). This is why ray tracing works!

### The BRDF

The **Bidirectional Reflectance Distribution Function**
$f_r(\mathbf{x}, \omega_i, \omega_o)$ describes how much light arriving
from direction $\omega_i$ is reflected toward $\omega_o$ at surface point
$\mathbf{x}$.

- **Lambertian (diffuse)**: $f_r = k_d / \pi$ — reflects equally in all
  directions. This is the "matte" assumption underlying photometric
  consistency in stereo, MVS, and direct SLAM
- **Specular**: concentrated reflection toward the mirror direction

### The Rendering Equation (Kajiya 1986)

$$
L_o(\mathbf{x}, \omega_o) = L_e(\mathbf{x}, \omega_o)
+ \int_{\Omega} f_r(\mathbf{x}, \omega_i, \omega_o) \, L_i(\mathbf{x}, \omega_i)
\, \cos\theta_i \, d\omega_i
$$

Outgoing = emitted + integral of (BRDF × incoming × cosine).

**NeRF's volume rendering** is the *participating media* generalisation:
instead of surfaces reflecting light, volumetric media emits and absorbs
light continuously along the ray.

### From Scene to Pixel

$$
E = \frac{\pi}{4} \left(\frac{D}{f}\right)^2 L \cos^4\theta
$$

Then:

$$
I = f_{\text{CRF}}\!\left(t_{\text{exp}} \cdot V(\mathbf{x}) \cdot E\right)
$$

where $V(\mathbf{x})$ is vignetting and $f_{\text{CRF}}$ is the camera
response function. **DSO's photometric calibration** (NB 07) estimates
exactly $(t, V, f_{\text{CRF}})$ to make photometric consistency valid.

---
## 1. The Volume Rendering Equation (NeRF)

### Physical Model

Consider a ray $\mathbf{r}(t) = \mathbf{o} + t\mathbf{d}$ passing through a participating medium.
The medium is described by:

- **Volume density** $\sigma(\mathbf{x}) \geq 0$: probability of a ray being absorbed per unit length at point $\mathbf{x}$
- **Emitted radiance** $\mathbf{c}(\mathbf{x}, \mathbf{d}) \in [0,1]^3$: color emitted at $\mathbf{x}$ in direction $\mathbf{d}$

### Transmittance (Beer-Lambert Law)

The probability that a ray travels from $t_n$ to $t$ without being absorbed:

$$
\boxed{T(t) = \exp\left(-\int_{t_n}^{t} \sigma(\mathbf{r}(s)) \, ds\right)}
$$

This is the **Beer-Lambert law** from optics — the fundamental equation governing
light attenuation in participating media. It arises from the ODE
$dT/dt = -\sigma(t) \, T(t)$ (the probability of *not* being absorbed in
$[t, t+dt]$ is $(1 - \sigma \, dt)$, giving the exponential decay solution above).

**Derivation**: Let $T(t)$ be the probability of no interaction in $[t_n, t]$.
In the interval $[t, t+dt]$, the probability of absorption is $\sigma(t) \, dt$,
so $T(t+dt) = T(t)(1 - \sigma(t) \, dt)$. Rearranging:

$$
\frac{T(t+dt) - T(t)}{dt} = -\sigma(t) \, T(t) \quad \Longrightarrow \quad
\frac{dT}{dt} = -\sigma(t) \, T(t)
$$

This separable ODE with initial condition $T(t_n) = 1$ has the unique solution above.

Properties:
- $T(t_n) = 1$ (starts at full transmittance)
- $T(t)$ is monotonically decreasing (since $\sigma \geq 0$)
- $T(t) \to 0$ as $t \to \infty$ (for non-trivial $\sigma$)

### Continuous Volume Rendering Equation

The expected color $C(\mathbf{r})$ accumulated along the ray:

$$
\boxed{C(\mathbf{r}) = \int_{t_n}^{t_f} T(t) \cdot \sigma(\mathbf{r}(t)) \cdot \mathbf{c}(\mathbf{r}(t), \mathbf{d}) \, dt}
$$

**Intuition**: At each point along the ray, the contribution to the final color is:
- $T(t)$: how much light from this point reaches the camera (not blocked by earlier stuff)
- $\sigma(t)$: how much "stuff" is here (density)
- $\mathbf{c}(t)$: what color is this stuff

Note that $T(t) \cdot \sigma(t) \, dt$ can be interpreted as the probability of the ray
terminating in the interval $[t, t + dt]$.

### Discrete Approximation (Numerical Quadrature)

Partition $[t_n, t_f]$ into $N$ intervals with sample points $t_1 < t_2 < \cdots < t_N$
and interval widths $\delta_i = t_{i+1} - t_i$.

Assuming $\sigma$ and $\mathbf{c}$ are constant within each interval:

$$
\boxed{\hat{C}(\mathbf{r}) = \sum_{i=1}^{N} T_i \cdot \alpha_i \cdot \mathbf{c}_i}
$$

where:

$$
\alpha_i = 1 - \exp(-\sigma_i \, \delta_i) \quad \text{(opacity of interval $i$)}
$$

$$
T_i = \prod_{j=1}^{i-1} (1 - \alpha_j) = \exp\left(-\sum_{j=1}^{i-1} \sigma_j \delta_j\right) \quad \text{(accumulated transmittance)}
$$

### Derivation of the Discrete Form

Starting from the continuous equation:

$$
\begin{aligned}
C &= \int_{t_n}^{t_f} T(t) \, \sigma(t) \, \mathbf{c}(t) \, dt \\
&\approx \sum_{i=1}^{N} \underbrace{\exp\left(-\int_{t_n}^{t_i} \sigma(s) ds\right)}_{T_i} \cdot \underbrace{\int_{t_i}^{t_{i+1}} \sigma(s) ds}_{\approx \sigma_i \delta_i} \cdot \mathbf{c}_i
\end{aligned}
$$

But more precisely, the contribution of interval $[t_i, t_{i+1}]$ is:

$$
\Delta C_i = T_i \cdot \int_{t_i}^{t_{i+1}} \sigma(s) \exp\left(-\int_{t_i}^{s} \sigma(u) du\right) ds \cdot \mathbf{c}_i
$$

With constant $\sigma_i$ in the interval:

$$
\int_{t_i}^{t_{i+1}} \sigma_i \exp(-\sigma_i (s - t_i)) ds = 1 - \exp(-\sigma_i \delta_i) = \alpha_i
$$

This is the alpha (opacity) of interval $i$. The final formula $\hat{C} = \sum_i T_i \alpha_i \mathbf{c}_i$ follows.

In [ ]:
# ── Volume Rendering Implementation ──

def volume_render(sigmas, colors, deltas):
    """
    Discrete volume rendering.
    
    Parameters
    ----------
    sigmas : (N,) array — volume density at each sample
    colors : (N, 3) array — RGB color at each sample
    deltas : (N,) array — interval widths
    
    Returns
    -------
    C : (3,) array — rendered pixel color
    weights : (N,) array — contribution weight of each sample
    T : (N,) array — transmittance at each sample
    alphas : (N,) array — opacity of each interval
    """
    alphas = 1.0 - np.exp(-sigmas * deltas)
    
    # T_i = prod_{j<i} (1 - alpha_j)
    T = np.ones_like(sigmas)
    for i in range(1, len(sigmas)):
        T[i] = T[i-1] * (1.0 - alphas[i-1])
    
    weights = T * alphas
    C = np.sum(weights[:, None] * colors, axis=0)
    
    return C, weights, T, alphas

In [ ]:
# ── Synthetic 1D scene: ray marching visualization ──

np.random.seed(42)

N_SAMPLES = 200
t_near, t_far = 0.0, 10.0
t_vals = np.linspace(t_near, t_far, N_SAMPLES)
deltas = np.diff(t_vals, append=t_vals[-1] + (t_vals[-1] - t_vals[-2]))

# Synthetic scene: two "objects" (Gaussian density bumps)
sigma_scene = (
    5.0 * np.exp(-0.5 * ((t_vals - 3.0) / 0.4)**2) +  # Object 1 (near, red)
    8.0 * np.exp(-0.5 * ((t_vals - 7.0) / 0.6)**2)     # Object 2 (far, blue)
)

colors_scene = np.zeros((N_SAMPLES, 3))
# Object 1: reddish
mask1 = np.exp(-0.5 * ((t_vals - 3.0) / 0.8)**2)
# Object 2: bluish
mask2 = np.exp(-0.5 * ((t_vals - 7.0) / 1.0)**2)
total = mask1 + mask2 + 1e-8
colors_scene[:, 0] = 0.9 * mask1 / total + 0.1 * mask2 / total  # R
colors_scene[:, 1] = 0.2 * mask1 / total + 0.2 * mask2 / total  # G
colors_scene[:, 2] = 0.1 * mask1 / total + 0.9 * mask2 / total  # B

# Render
C_rendered, weights, T, alphas = volume_render(sigma_scene, colors_scene, deltas)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Density profile
axes[0, 0].fill_between(t_vals, sigma_scene, alpha=0.3, color='purple')
axes[0, 0].plot(t_vals, sigma_scene, 'purple', lw=2)
axes[0, 0].set_xlabel('t (ray parameter)'); axes[0, 0].set_ylabel('$\\sigma(t)$')
axes[0, 0].set_title('Volume Density $\\sigma(t)$'); axes[0, 0].grid(True)

# Transmittance
axes[0, 1].plot(t_vals, T, 'g-', lw=2)
axes[0, 1].fill_between(t_vals, T, alpha=0.2, color='green')
axes[0, 1].set_xlabel('t'); axes[0, 1].set_ylabel('$T(t)$')
axes[0, 1].set_title('Transmittance $T(t) = \\exp(-\\int \\sigma \\, ds)$')
axes[0, 1].set_ylim(0, 1.05); axes[0, 1].grid(True)

# Weights (contribution of each sample)
axes[1, 0].fill_between(t_vals, weights, alpha=0.4, color='orange')
axes[1, 0].plot(t_vals, weights, 'orange', lw=2)
axes[1, 0].set_xlabel('t'); axes[1, 0].set_ylabel('$w_i = T_i \\cdot \\alpha_i$')
axes[1, 0].set_title('Sample Weights (where color is accumulated)')
axes[1, 0].grid(True)

# Color along ray and final rendered color
for ch, name, color in [(0, 'R', 'red'), (1, 'G', 'green'), (2, 'B', 'blue')]:
    axes[1, 1].plot(t_vals, colors_scene[:, ch], '--', color=color, alpha=0.5, label=f'{name} (scene)')
axes[1, 1].axhline(C_rendered[0], color='red', lw=2, label=f'Rendered R={C_rendered[0]:.2f}')
axes[1, 1].axhline(C_rendered[1], color='green', lw=2, label=f'Rendered G={C_rendered[1]:.2f}')
axes[1, 1].axhline(C_rendered[2], color='blue', lw=2, label=f'Rendered B={C_rendered[2]:.2f}')
axes[1, 1].set_xlabel('t'); axes[1, 1].set_ylabel('Color')
axes[1, 1].set_title(f'Rendered Color: ({C_rendered[0]:.2f}, {C_rendered[1]:.2f}, {C_rendered[2]:.2f})')
axes[1, 1].legend(fontsize=8); axes[1, 1].grid(True)

plt.suptitle('Volume Rendering: Ray Marching Through a Synthetic Scene', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Ray marching: render a 2D image of a synthetic 3D scene ──

np.random.seed(2024)

# Simple pinhole camera
IMG_H, IMG_W = 64, 64
FOCAL = 60.0
CX, CY = IMG_W / 2, IMG_H / 2

# Scene: two colored spheres
spheres = [
    {'center': np.array([-1.0, 0.0, 5.0]), 'radius': 1.0, 'color': np.array([0.9, 0.2, 0.1]), 'density': 15.0},
    {'center': np.array([1.0, 0.5, 7.0]), 'radius': 1.2, 'color': np.array([0.1, 0.3, 0.9]), 'density': 12.0},
]

def scene_density_color(pts):
    """Query scene density and color at 3D points."""
    sigma = np.zeros(len(pts))
    color = np.ones((len(pts), 3)) * 0.8  # background grayish
    
    for s in spheres:
        dist = norm(pts - s['center'], axis=1)
        sphere_sigma = s['density'] * np.exp(-0.5 * (dist / (s['radius'] * 0.5))**2)
        mask = sphere_sigma > 0.01
        weight = sphere_sigma / (sigma + sphere_sigma + 1e-10)
        color[mask] = weight[mask, None] * s['color'] + (1 - weight[mask, None]) * color[mask]
        sigma += sphere_sigma
    
    return sigma, color

# Render image via ray marching
N_RAY_SAMPLES = 64
t_near_r, t_far_r = 2.0, 12.0
t_samples = np.linspace(t_near_r, t_far_r, N_RAY_SAMPLES)
dt = (t_far_r - t_near_r) / N_RAY_SAMPLES

image = np.zeros((IMG_H, IMG_W, 3))
depth_map = np.zeros((IMG_H, IMG_W))

for v in range(IMG_H):
    # Batch all pixels in this row
    rays_d = np.zeros((IMG_W, 3))
    for u in range(IMG_W):
        rays_d[u] = np.array([(u - CX) / FOCAL, (v - CY) / FOCAL, 1.0])
        rays_d[u] /= norm(rays_d[u])
    
    rays_o = np.zeros((IMG_W, 3))  # Camera at origin
    
    for u in range(IMG_W):
        pts = rays_o[u] + t_samples[:, None] * rays_d[u]  # (N_RAY_SAMPLES, 3)
        sigmas, colors = scene_density_color(pts)
        deltas_r = np.full(N_RAY_SAMPLES, dt)
        
        C, weights, _, _ = volume_render(sigmas, colors, deltas_r)
        image[v, u] = np.clip(C, 0, 1)
        depth_map[v, u] = np.sum(weights * t_samples)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image)
axes[0].set_title('Volume Rendered Image')
axes[0].axis('off')

depth_vis = axes[1].imshow(depth_map, cmap='plasma')
axes[1].set_title('Expected Depth')
axes[1].axis('off')
plt.colorbar(depth_vis, ax=axes[1], fraction=0.046)

plt.suptitle('NeRF-Style Ray Marching: Rendered Image + Depth', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Image shape: {image.shape}, rendered {IMG_H * IMG_W} rays × {N_RAY_SAMPLES} samples each.")

# --- 3D Point Cloud from Back-Projected Depth + 3D Density Field ---
v_grid, u_grid = np.mgrid[0:IMG_H, 0:IMG_W]

Z_pc = depth_map
X_pc = (u_grid - CX) * Z_pc / FOCAL
Y_pc = (v_grid - CY) * Z_pc / FOCAL

valid_mask = Z_pc > (t_near_r + 0.5)
step_pc = 1

fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(
    X_pc[valid_mask][::step_pc],
    Y_pc[valid_mask][::step_pc],
    Z_pc[valid_mask][::step_pc],
    c=image[valid_mask][::step_pc],
    s=4, alpha=0.7
)
ax1.set_xlabel('X (world)')
ax1.set_ylabel('Y (world)')
ax1.set_zlabel('Z — Depth')
ax1.set_title('Scene Point Cloud\n(back-projected from rendered depth, colored by RGB)')
ax1.view_init(elev=30, azim=-45)

ax2 = fig.add_subplot(122, projection='3d')
grid_res = 25
xs = np.linspace(-3, 3, grid_res)
ys = np.linspace(-2, 2, grid_res)
zs = np.linspace(t_near_r, t_far_r, grid_res)
xx, yy, zz = np.meshgrid(xs, ys, zs, indexing='ij')
grid_pts = np.column_stack([xx.ravel(), yy.ravel(), zz.ravel()])
sigma_grid, color_grid = scene_density_color(grid_pts)

dense_mask = sigma_grid > 0.5
if np.sum(dense_mask) > 0:
    sc2 = ax2.scatter(
        grid_pts[dense_mask, 0],
        grid_pts[dense_mask, 1],
        grid_pts[dense_mask, 2],
        c=color_grid[dense_mask],
        s=np.clip(sigma_grid[dense_mask] * 3, 2, 40),
        alpha=0.5
    )
ax2.set_xlabel('X (world)')
ax2.set_ylabel('Y (world)')
ax2.set_zlabel('Z (world)')
ax2.set_title(f'3D Density Field (σ > 0.5)\n{np.sum(dense_mask)} / {len(sigma_grid)} voxels occupied')
ax2.view_init(elev=30, azim=-45)

plt.suptitle('3D Views: NeRF Scene — Point Cloud & Implicit Density Field', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. NeRF Architecture

### The MLP

NeRF represents a scene as a continuous function parameterized by an MLP:

$$
F_\theta : (\mathbf{x}, \mathbf{d}) \to (\sigma, \mathbf{c})
$$

where $\mathbf{x} = (x, y, z)$ is the 3D position and $\mathbf{d} = (\theta, \phi)$ is the viewing direction.

Key design: $\sigma$ depends only on $\mathbf{x}$ (geometry is view-independent), while
$\mathbf{c}$ depends on both $\mathbf{x}$ and $\mathbf{d}$ (appearance is view-dependent).

### Positional Encoding

MLPs are biased toward learning low-frequency functions. **Positional encoding** maps
inputs to a higher-dimensional space:

$$
\gamma(p) = \left(\sin(2^0 \pi p), \cos(2^0 \pi p), \ldots, \sin(2^{L-1} \pi p), \cos(2^{L-1} \pi p)\right)
$$

Applied independently to each coordinate:
- Position: $\gamma(\mathbf{x})$ with $L = 10$ → 60-dimensional encoding
- Direction: $\gamma(\mathbf{d})$ with $L = 4$ → 24-dimensional encoding

This enables the MLP to represent high-frequency detail.

### Hierarchical Sampling

Two-pass rendering for efficiency:

1. **Coarse network**: Evaluate at $N_c$ stratified random samples along each ray.
   Compute weights $w_i = T_i \alpha_i$.

2. **Fine network**: Sample $N_f$ additional points from the distribution defined by
   the coarse weights (inverse CDF sampling). This concentrates samples near surfaces.

$$
\hat{t}_j \sim \text{PDF}(t) = \frac{w(t)}{\sum_i w_i}
$$

### Training Loss

$$
\mathcal{L} = \sum_{\mathbf{r} \in \mathcal{R}} \left[ \| \hat{C}_c(\mathbf{r}) - C(\mathbf{r}) \|_2^2 + \| \hat{C}_f(\mathbf{r}) - C(\mathbf{r}) \|_2^2 \right]
$$

where $\mathcal{R}$ is a batch of rays and $C(\mathbf{r})$ is the ground-truth pixel color.

### Limitations

- **Slow training**: days per scene on a single GPU
- **Slow rendering**: per-pixel MLP evaluation (~30s for 800×800)
- **No explicit geometry**: hard to extract meshes
- **Per-scene overfitting**: a new model for each scene

---
## 3. 3D Gaussian Splatting — Full Forward Model

3D Gaussian Splatting (3DGS) represents the scene as a collection of anisotropic 3D Gaussians,
each with learnable parameters. Unlike NeRF, 3DGS uses an **explicit** representation that
enables real-time rendering via rasterization (not ray marching).

### Scene Representation

Each Gaussian $k$ is defined by:

| Parameter | Symbol | Size | Description |
|-----------|--------|------|-------------|
| Mean (center) | $\boldsymbol{\mu}_k$ | 3 | 3D position |
| Covariance | $\Sigma_k$ | 3×3 | Shape and orientation |
| Color | $\mathbf{c}_k$ | 3 (or SH coefficients) | Appearance |
| Opacity | $\alpha_k$ | 1 | Transparency |

The 3D Gaussian function:

$$
G_k(\mathbf{x}) = \exp\left(-\frac{1}{2} (\mathbf{x} - \boldsymbol{\mu}_k)^\top \Sigma_k^{-1} (\mathbf{x} - \boldsymbol{\mu}_k)\right)
$$

### Covariance Parameterization

To ensure $\Sigma_k$ is always positive semi-definite, it is parameterized via a rotation
matrix $R_k$ and a diagonal scale matrix $S_k$:

$$
\boxed{\Sigma_k = R_k \, S_k \, S_k^\top \, R_k^\top}
$$

where $S_k = \text{diag}(s_1, s_2, s_3)$ contains the scales along principal axes,
and $R_k$ is stored as a unit quaternion $\mathbf{q}_k \in \mathbb{R}^4$.

This decomposition guarantees PSD (positive semi-definite) for any $R_k, S_k$ values,
which is crucial for gradient-based optimization.

### Projection to 2D: EWA Splatting

To render a Gaussian onto a 2D image, we project it using the **EWA (Elliptical Weighted Average)** framework (Zwicker et al., 2002).

Given world-to-camera transform $W$ (the view matrix) and the Jacobian $J$ of the
perspective projection at the Gaussian's center:

$$
\boxed{\Sigma_{2D} = J \, W \, \Sigma_k \, W^\top \, J^\top}
$$

where:
- $W \in \mathbb{R}^{3 \times 3}$ is the rotational part of the view matrix
- $J \in \mathbb{R}^{2 \times 3}$ is the Jacobian of the projective mapping

For a point at camera coordinates $(x, y, z)$, the projective Jacobian is:

$$
J = \frac{\partial}{\partial \mathbf{p}_c} \begin{pmatrix} f_x \frac{x}{z} + c_x \\ f_y \frac{y}{z} + c_y \end{pmatrix} = \begin{bmatrix} \frac{f_x}{z} & 0 & -\frac{f_x x}{z^2} \\ 0 & \frac{f_y}{z} & -\frac{f_y y}{z^2} \end{bmatrix}
$$

The projected 2D Gaussian evaluated at pixel $\mathbf{u} = (u, v)$:

$$
G_k^{2D}(\mathbf{u}) = \exp\left(-\frac{1}{2} (\mathbf{u} - \boldsymbol{\mu}_k^{2D})^\top \Sigma_{2D}^{-1} (\mathbf{u} - \boldsymbol{\mu}_k^{2D})\right)
$$

where $\boldsymbol{\mu}_k^{2D}$ is the projected center.

### Alpha Compositing (Front-to-Back)

Sort Gaussians by depth (front to back). For each pixel, accumulate color:

$$
\boxed{C(\mathbf{u}) = \sum_{i=1}^{N} \mathbf{c}_i \cdot \alpha_i \cdot G_i^{2D}(\mathbf{u}) \cdot T_i}
$$

$$
T_i = \prod_{j=1}^{i-1} \left(1 - \alpha_j \cdot G_j^{2D}(\mathbf{u})\right)
$$

This is essentially the same alpha-compositing as NeRF's discrete volume rendering,
but applied to projected 2D Gaussians rather than ray samples.

### Training Loss

$$
\boxed{\mathcal{L} = (1 - \lambda) \cdot \mathcal{L}_1 + \lambda \cdot (1 - \text{SSIM})}
$$

where $\lambda = 0.2$ (default), $\mathcal{L}_1 = \|\hat{I} - I_{\text{gt}}\|_1$, and
SSIM measures structural similarity.

### Adaptive Density Control

During training, 3DGS dynamically manages the number of Gaussians based on the
**positional gradient magnitude** accumulated over several iterations.

Let $\bar{\nabla}_{\mu_k} = \frac{1}{M}\sum_{m=1}^{M} \|\nabla_{\mu_k^{2D}} \mathcal{L}_m\|$
be the mean view-space positional gradient for Gaussian $k$ over $M$ recent training steps.
Densification is triggered when $\bar{\nabla}_{\mu_k} > \tau_{\text{pos}}$ (default $\tau_{\text{pos}} = 0.0002$).

The clone-vs-split decision depends on the Gaussian's spatial extent $\max(s_1, s_2, s_3)$ relative to a scene-dependent threshold $\tau_S$:

- **Clone** ($\max(s) < \tau_S$): the Gaussian is **small but under-reconstructed** — duplicate it and move the copy along the positional gradient direction. This fills gaps in under-represented regions.
- **Split** ($\max(s) \geq \tau_S$): the Gaussian is **large and covers too much structure** — replace it with two Gaussians of scale $s / \phi$ (default $\phi = 1.6$), sampling their positions from the original's distribution $\mathcal{N}(\mu_k, \Sigma_k)$.

Pruning removes Gaussians that contribute negligibly:
- **Transparent**: $\alpha_k < \epsilon_\alpha$ (default $\epsilon_\alpha = 0.005$) — nearly invisible
- **Too large**: Gaussians whose 2D projected area exceeds a screen-space threshold — prevents "floaters"
- **Periodic opacity reset**: every $N_{\text{reset}}$ iterations (default 3000), all opacities are set to a low value and must be re-learned, flushing transient Gaussians

> ### Spherical Harmonics for View-Dependent Color
>
> 3DGS represents each Gaussian's color as a function of viewing direction using **spherical harmonics (SH)**. The real SH basis functions up to degree $l = 2$ are:
>
> | $l$ | $m$ | $Y_l^m(\theta, \phi)$ | Name |
> |-----|-----|-----------------------|------|
> | 0 | 0 | $\frac{1}{2\sqrt{\pi}}$ | DC (constant) |
> | 1 | -1 | $\frac{\sqrt{3}}{2\sqrt{\pi}} y$ | Linear Y |
> | 1 | 0 | $\frac{\sqrt{3}}{2\sqrt{\pi}} z$ | Linear Z |
> | 1 | 1 | $\frac{\sqrt{3}}{2\sqrt{\pi}} x$ | Linear X |
> | 2 | -2 | $\frac{\sqrt{15}}{2\sqrt{\pi}} xy$ | Quadratic |
> | 2 | -1 | $\frac{\sqrt{15}}{2\sqrt{\pi}} yz$ | Quadratic |
> | 2 | 0 | $\frac{\sqrt{5}}{4\sqrt{\pi}} (3z^2 - 1)$ | Quadratic |
> | 2 | 1 | $\frac{\sqrt{15}}{2\sqrt{\pi}} xz$ | Quadratic |
> | 2 | 2 | $\frac{\sqrt{15}}{4\sqrt{\pi}} (x^2 - y^2)$ | Quadratic |
>
> where $(x, y, z) = (\sin\theta\cos\phi, \sin\theta\sin\phi, \cos\theta)$ is the unit viewing direction.
>
> Each Gaussian stores SH coefficients $c_{lm} \in \mathbb{R}^3$ (one per RGB channel) up to some maximum degree $l_{\max}$ (typically 3, giving $(l_{\max}+1)^2 = 16$ coefficients per channel). The color seen from direction $\mathbf{d}$ is:
>
> $$\mathbf{c}(\mathbf{d}) = \sum_{l=0}^{l_{\max}} \sum_{m=-l}^{l} c_{lm} \, Y_l^m(\mathbf{d})$$
>
> The $l=0$ term gives the base (Lambertian) color; higher orders capture specular highlights and view-dependent effects. During optimization, SH coefficients are progressively activated (first $l=0$, then $l\leq 1$, etc.) to avoid overfitting early in training.

In [ ]:
# ── 3D Gaussian Splatting: From-Scratch Implementation ──

class Gaussian3D:
    """A single 3D Gaussian splat."""
    def __init__(self, mu, scales, quat, color, opacity):
        self.mu = np.array(mu, dtype=np.float64)       # (3,) center
        self.scales = np.array(scales, dtype=np.float64) # (3,) scales
        self.quat = np.array(quat, dtype=np.float64)    # (4,) quaternion
        self.color = np.array(color, dtype=np.float64)   # (3,) RGB
        self.opacity = float(opacity)                     # scalar alpha
    
    @property
    def rotation_matrix(self):
        return Rotation.from_quat(self.quat[[1, 2, 3, 0]]).as_matrix()
    
    @property
    def covariance_3d(self):
        """Σ = R S S^T R^T"""
        R = self.rotation_matrix
        S = np.diag(self.scales)
        return R @ S @ S.T @ R.T


def project_gaussian(gauss, T_cw, K):
    """
    Project a 3D Gaussian to 2D using EWA splatting.
    
    Parameters
    ----------
    gauss : Gaussian3D
    T_cw : (4, 4) camera-from-world transform
    K : (3, 3) intrinsic matrix
    
    Returns
    -------
    mu_2d : (2,) projected center
    cov_2d : (2, 2) projected covariance
    depth : float — depth of center
    """
    # Transform center to camera frame
    R_cw = T_cw[:3, :3]
    t_cw = T_cw[:3, 3]
    p_cam = R_cw @ gauss.mu + t_cw
    
    if p_cam[2] <= 0.1:
        return None, None, None
    
    # Project center to image
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    mu_2d = np.array([fx * p_cam[0] / p_cam[2] + cx,
                      fy * p_cam[1] / p_cam[2] + cy])
    
    # Jacobian of perspective projection
    x, y, z = p_cam
    J = np.array([
        [fx / z, 0, -fx * x / z**2],
        [0, fy / z, -fy * y / z**2]
    ])
    
    # Project 3D covariance to 2D: Σ_2d = J W Σ W^T J^T
    W = R_cw
    Sigma_3d = gauss.covariance_3d
    cov_2d = J @ W @ Sigma_3d @ W.T @ J.T
    
    return mu_2d, cov_2d, p_cam[2]


def eval_gaussian_2d(pixel, mu_2d, cov_2d_inv):
    """Evaluate 2D Gaussian at a pixel location."""
    d = pixel - mu_2d
    return np.exp(-0.5 * d @ cov_2d_inv @ d)


def render_gaussians(gaussians, T_cw, K, img_h, img_w, bg_color=np.array([1.0, 1.0, 1.0])):
    """
    Render a set of 3D Gaussians to an image.
    
    Full forward model:
    1. Project each Gaussian to 2D (EWA splatting)
    2. Sort by depth (front to back)
    3. Alpha-composite for each pixel
    """
    # Project all Gaussians
    projected = []
    for g in gaussians:
        mu_2d, cov_2d, depth = project_gaussian(g, T_cw, K)
        if mu_2d is None:
            continue
        # Skip if too far from image
        if mu_2d[0] < -50 or mu_2d[0] > img_w + 50 or mu_2d[1] < -50 or mu_2d[1] > img_h + 50:
            continue
        try:
            cov_2d_inv = inv(cov_2d + np.eye(2) * 0.3)  # Small regularization
        except np.linalg.LinAlgError:
            continue
        projected.append((depth, mu_2d, cov_2d, cov_2d_inv, g.color, g.opacity))
    
    # Sort by depth (front to back)
    projected.sort(key=lambda x: x[0])
    
    # Render
    image = np.zeros((img_h, img_w, 3))
    
    for v in range(img_h):
        for u in range(img_w):
            pixel = np.array([u + 0.5, v + 0.5])
            T_acc = 1.0  # Accumulated transmittance
            C_acc = np.zeros(3)
            
            for (depth, mu_2d, cov_2d, cov_2d_inv, color, opacity) in projected:
                if T_acc < 0.001:
                    break
                
                G_val = eval_gaussian_2d(pixel, mu_2d, cov_2d_inv)
                a = opacity * G_val
                
                if a < 1.0 / 255.0:
                    continue
                
                C_acc += color * a * T_acc
                T_acc *= (1.0 - a)
            
            # Background
            C_acc += bg_color * T_acc
            image[v, u] = np.clip(C_acc, 0, 1)
    
    return image

In [ ]:
# ── Create and render a synthetic 3DGS scene ──

np.random.seed(42)

# Create synthetic Gaussians
gaussians = []

# Ground plane (flat Gaussians)
for x in np.linspace(-3, 3, 7):
    for z in np.linspace(4, 10, 7):
        gaussians.append(Gaussian3D(
            mu=[x, 1.5, z],
            scales=[0.5, 0.02, 0.5],
            quat=[1, 0, 0, 0],
            color=[0.3, 0.7, 0.3],
            opacity=0.9
        ))

# Red sphere (cluster of Gaussians)
for _ in range(20):
    offset = np.random.randn(3) * 0.3
    q = Rotation.random().as_quat()  # [x,y,z,w]
    q = np.array([q[3], q[0], q[1], q[2]])  # [w,x,y,z]
    gaussians.append(Gaussian3D(
        mu=np.array([-1.0, 0.0, 6.0]) + offset,
        scales=np.random.uniform(0.1, 0.4, 3),
        quat=q,
        color=[0.9, 0.15, 0.1],
        opacity=0.85
    ))

# Blue sphere
for _ in range(20):
    offset = np.random.randn(3) * 0.35
    q = Rotation.random().as_quat()
    q = np.array([q[3], q[0], q[1], q[2]])
    gaussians.append(Gaussian3D(
        mu=np.array([1.5, -0.2, 7.5]) + offset,
        scales=np.random.uniform(0.1, 0.5, 3),
        quat=q,
        color=[0.1, 0.2, 0.9],
        opacity=0.8
    ))

# Yellow ellipsoid
for _ in range(15):
    offset = np.random.randn(3) * 0.25
    q = Rotation.from_euler('z', np.random.uniform(0, 2*np.pi)).as_quat()
    q = np.array([q[3], q[0], q[1], q[2]])
    gaussians.append(Gaussian3D(
        mu=np.array([0.0, -0.5, 5.0]) + offset,
        scales=[0.6, 0.2, 0.3],
        quat=q,
        color=[0.95, 0.85, 0.1],
        opacity=0.75
    ))

print(f"Scene: {len(gaussians)} Gaussians")

# Camera setup
K_gs = np.array([[120, 0, 64], [0, 120, 48], [0, 0, 1]], dtype=np.float64)
T_cw = np.eye(4)  # Camera at origin looking along +Z

print("Rendering... (this is a pure-Python reference, not GPU-accelerated)")
img_gs = render_gaussians(gaussians, T_cw, K_gs, img_h=96, img_w=128)

fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.imshow(img_gs)
ax.set_title(f'3D Gaussian Splatting Rendered Image\n({len(gaussians)} Gaussians, pure Python)', fontsize=13)
ax.axis('off')
plt.tight_layout(); plt.show()

# --- 3D View of Gaussian Splat Centers ---
centers = np.array([g.mu for g in gaussians])
colors_3d = np.clip(np.array([g.color for g in gaussians]), 0, 1)
opacities_3d = np.array([g.opacity for g in gaussians])
max_scales = np.array([np.max(g.scales) for g in gaussians])

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(
    centers[:, 0], centers[:, 1], centers[:, 2],
    c=colors_3d,
    s=opacities_3d * 40,
    alpha=0.7
)
ax1.set_xlabel('X (world)')
ax1.set_ylabel('Y (world)')
ax1.set_zlabel('Z (world)')
ax1.set_title(f'3DGS: Gaussian Centers\n({len(gaussians)} splats, color = learned RGB)')
ax1.view_init(elev=30, azim=-45)

ax2 = fig.add_subplot(122, projection='3d')
sc2 = ax2.scatter(
    centers[:, 0], centers[:, 1], centers[:, 2],
    c=max_scales,
    cmap='viridis',
    s=opacities_3d * 40,
    alpha=0.7
)
ax2.set_xlabel('X (world)')
ax2.set_ylabel('Y (world)')
ax2.set_zlabel('Z (world)')
ax2.set_title(f'3DGS: Gaussian Centers\n(color = max scale per splat)')
ax2.view_init(elev=30, azim=-45)
fig.colorbar(sc2, ax=ax2, shrink=0.5, label='Max scale')

plt.suptitle('3D Gaussian Splatting — Scene Structure in 3D', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Multi-view rendering ──

def make_camera_pose(angle_y, distance=0, height=0):
    """Create a camera pose looking at the scene center."""
    R = Rotation.from_euler('y', angle_y).as_matrix()
    t = np.array([np.sin(angle_y) * distance, height, -np.cos(angle_y) * distance + distance])
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T

# Render from 3 viewpoints
angles = [-0.3, 0.0, 0.3]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, angle in enumerate(angles):
    T = make_camera_pose(angle, distance=1.5, height=0.0)
    img = render_gaussians(gaussians, T, K_gs, img_h=72, img_w=96)
    axes[idx].imshow(img)
    axes[idx].set_title(f'View angle: {np.degrees(angle):.0f}°')
    axes[idx].axis('off')

plt.suptitle('3DGS: Novel View Synthesis from Multiple Viewpoints', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 4. Differentiable Rendering

### Why 3DGS is Differentiable

The key advantage of 3DGS over traditional rasterization: every step in the rendering
pipeline is differentiable, enabling gradient-based optimization of all Gaussian parameters.

Recall the rendered color at pixel $\mathbf{u}$:

$$
C(\mathbf{u}) = \sum_{i} \mathbf{c}_i \cdot \alpha_i \cdot G_i^{2D}(\mathbf{u}) \cdot T_i
$$

### Gradient w.r.t. Mean $\boldsymbol{\mu}_k$

The mean affects both the projected center $\boldsymbol{\mu}_k^{2D}$ (which affects $G_k^{2D}$)
and the depth (which affects the sorting order, but this is handled by approximation).

$$
\frac{\partial C}{\partial \boldsymbol{\mu}_k} = \frac{\partial C}{\partial G_k^{2D}} \cdot \frac{\partial G_k^{2D}}{\partial \boldsymbol{\mu}_k^{2D}} \cdot \frac{\partial \boldsymbol{\mu}_k^{2D}}{\partial \boldsymbol{\mu}_k}
$$

Each term is analytically differentiable:

$$
\frac{\partial G_k^{2D}}{\partial \boldsymbol{\mu}_k^{2D}} = G_k^{2D}(\mathbf{u}) \cdot \Sigma_{2D}^{-1} (\mathbf{u} - \boldsymbol{\mu}_k^{2D})
$$

$$
\frac{\partial \boldsymbol{\mu}_k^{2D}}{\partial \boldsymbol{\mu}_k} = J_\pi \cdot R_{cw}
$$

### Gradient w.r.t. Covariance $\Sigma_k$

The 3D covariance affects the 2D projected covariance:

$$
\frac{\partial C}{\partial \Sigma_k} = \frac{\partial C}{\partial \Sigma_{2D}} \cdot \frac{\partial \Sigma_{2D}}{\partial \Sigma_k}
$$

Since $\Sigma_{2D} = J W \Sigma_k W^\top J^\top$, the chain rule gives us gradients
that flow back through the projection to the 3D parameters.

Through the $\Sigma = R S S^\top R^\top$ parameterization, gradients further flow to
the quaternion $\mathbf{q}_k$ and scales $\mathbf{s}_k$.

### Gradient w.r.t. Color and Opacity

These are straightforward from the compositing equation:

$$
\frac{\partial C}{\partial \mathbf{c}_k} = \alpha_k \cdot G_k^{2D}(\mathbf{u}) \cdot T_k
$$

$$
\frac{\partial C}{\partial \alpha_k} = \mathbf{c}_k \cdot G_k^{2D}(\mathbf{u}) \cdot T_k + \text{(effect on later Gaussians' } T_i \text{)}
$$

### Key Insight

Unlike mesh-based rendering (where gradients are zero almost everywhere due to
rasterization discontinuities), 3DGS produces **smooth gradients** because:
1. Gaussians have infinite support (soft boundaries)
2. The alpha compositing is a continuous function
3. The covariance parameterization is smooth

---
## 6. MASt3R-SLAM

### Foundation Model Meets SLAM

**MASt3R** (Matching and Stereo 3D Reconstruction) is a foundation model that directly
predicts 3D pointmaps from image pairs, eliminating the traditional pipeline of feature
extraction → matching → triangulation.

### How It Works

Given two images $I_1, I_2$, MASt3R predicts:

$$
\text{MASt3R}(I_1, I_2) \to (\mathbf{X}_1, \mathbf{X}_2, \mathcal{C}_1, \mathcal{C}_2)
$$

where:
- $\mathbf{X}_1 \in \mathbb{R}^{H \times W \times 3}$: 3D pointmap for image 1 (in frame 1)
- $\mathbf{X}_2 \in \mathbb{R}^{H \times W \times 3}$: 3D pointmap for image 2 (in frame 1)
- $\mathcal{C}_1, \mathcal{C}_2$: confidence maps

### MASt3R-SLAM Architecture

1. **Frontend**: Run MASt3R on consecutive frame pairs to get pointmaps
2. **Pointmap alignment**: Align pointmaps across frames to get relative poses
3. **Backend**: Global alignment of all pointmaps (similar to pose graph optimization)
4. **Loop closure**: Re-run MASt3R on revisited pairs

### Key Innovations

- **No feature matching needed**: MASt3R directly produces correspondences
- **Camera-model agnostic**: Works with any camera (fisheye, pinhole, etc.)
- **Dense 3D from monocular**: Every pixel gets a 3D coordinate
- **Robust**: Trained on massive diverse data, handles challenging scenes

### Relation to DUSt3R

MASt3R builds on **DUSt3R** (Dense Unconstrained Stereo 3D Reconstruction):
- DUSt3R: Predicts pointmaps, no matching
- MASt3R: Adds local feature matching for better accuracy
- MASt3R-SLAM: Extends to sequential SLAM with loop closure

---
## 7. Neural Implicit Surfaces

### The Problem with NeRF's Geometry

NeRF represents geometry as volume density $\sigma$. This is great for rendering but
poor for extracting surfaces: the density field is often noisy and doesn't correspond
to a clean surface.

### Signed Distance Functions (SDF)

An SDF $f: \mathbb{R}^3 \to \mathbb{R}$ maps each point to its signed distance to the surface:

$$
f(\mathbf{x}) = \begin{cases}
-d(\mathbf{x}, \mathcal{S}) & \text{inside} \\
0 & \text{on surface } \mathcal{S} \\
+d(\mathbf{x}, \mathcal{S}) & \text{outside}
\end{cases}
$$

The surface is the zero level set: $\mathcal{S} = \{\mathbf{x} : f(\mathbf{x}) = 0\}$.

### NeuS (Wang et al., 2021)

**Key idea**: Convert SDF to density using a logistic function, then apply volume rendering.

The density is derived from the SDF $f$ via:

$$
\sigma(t) = \max\left(\frac{-\frac{d\Phi_s}{dt}(f(\mathbf{r}(t)))}{\Phi_s(f(\mathbf{r}(t)))}, 0\right)
$$

where $\Phi_s(x) = (1 + e^{-sx})^{-1}$ is the sigmoid with learnable sharpness $s$.

This ensures:
- Density peaks at the surface ($f = 0$)
- The weight function $w(t) = T(t)\sigma(t)$ is an **unbiased estimator** of the surface location

**Why unbiased?** For a straight ray passing through a flat surface, the weight $w(t) = T(t)\sigma(t)$ attains its maximum exactly at $f(\mathbf{r}(t^*)) = 0$ (the surface). Proof: $w(t) \propto \Phi_s'(f(t)) / \Phi_s(f(t)) \cdot \exp(-\int_0^t \sigma)$. Differentiating and using $\frac{d}{dt}\Phi_s(f) = \Phi_s' \cdot f'$, the maximum of $w$ occurs where $f = 0$, irrespective of the viewing angle. Standard NeRF's $\sigma = \text{ReLU}(\cdot)$ lacks this property — the maximum shifts with viewing direction, causing geometric bias.

- As $s \to \infty$, the weight concentrates on the exact surface (delta function limit)

**Eikonal loss** for SDF regularization:

$$
\mathcal{L}_{\text{eikonal}} = \sum_{\mathbf{x}} (\|\nabla f(\mathbf{x})\| - 1)^2
$$

This enforces the SDF property $\|\nabla f\| = 1$ everywhere.

### VolSDF (Yariv et al., 2021)

Alternative SDF-to-density mapping using the Laplace CDF:

$$
\sigma(\mathbf{x}) = \frac{1}{\beta} \Psi_\beta(-f(\mathbf{x})), \quad \Psi_\beta(s) = \begin{cases} \frac{1}{2}\exp(s/\beta) & s \leq 0 \\ 1 - \frac{1}{2}\exp(-s/\beta) & s > 0 \end{cases}
$$

where $\beta > 0$ controls surface sharpness.

### Comparison

| Method | Geometry | Rendering | Surface Quality |
|--------|----------|-----------|------------------|
| NeRF | Density $\sigma$ | Volume rendering | Poor |
| NeuS | SDF $f$ | Volume rendering with SDF-to-$\sigma$ | **Excellent** |
| VolSDF | SDF $f$ | Volume rendering with Laplace CDF | **Excellent** |
| 3DGS | Gaussian ellipsoids | Splatting | Medium (good with regularization) |

## 11. Dense SLAM Evolution

The trajectory from hand-crafted volumetric to learned explicit representations:

| System | Year | Representation | Key Innovation |
|--------|------|----------------|----------------|
| **KinectFusion** | 2011 | TSDF | First real-time dense RGB-D fusion |
| **ElasticFusion** | 2015 | Surfels | No pose graph — frame-to-model + non-rigid deformation |
| **BundleFusion** | 2017 | TSDF + hierarchy | Globally consistent dense reconstruction |
| **SplaTAM** | 2024 | 3D Gaussians | "Splat, Track & Map" — differentiable rendering for tracking |
| **EGG-Fusion** | 2025 | Gaussian surfels | Uncertainty-aware fusion, 0.6 cm error on Replica |

---
## 12. 3DGS-SLAM — Frontier Systems (2025–2026)

The 3DGS-SLAM space has exploded. Below are the key systems that push beyond SplaTAM / MonoGS
and define the 2025-2026 state of the art.

### MASt3R-SLAM (Murai, Dexheimer & Davison, CVPR 2025)

The first real-time SLAM built on a **Geometric Foundation Model** (see §6 above).
Instead of hand-crafted feature matching + PnP + bundle adjustment, MASt3R-SLAM runs
the MASt3R transformer end-to-end to produce dense 3D pointmaps and relative poses.

- **No feature extraction / matching / RANSAC pipeline** — a single forward pass replaces it
- **Camera-agnostic**: handles pinhole, fisheye, mixed-focal without calibration changes
- Core loop: pairwise MASt3R inference → pointmap alignment → global optimisation
- Real-time at ~15 FPS on an RTX 4090

### GLAM-SLAM (2026)

**Gaussian Latticework Anchored Mapping**: hybridises a classical feature-based frontend
with a structured Gaussian backend.

| Component | Choice |
|-----------|--------|
| Frontend | ORB-SLAM2 (feature tracking, loop closure) |
| Backend | **Structured Gaussian anchor grid** — Gaussians are tethered to a 3D lattice, preventing drift during optimisation |
| Key idea | Anchor grid acts as a spatial regulariser, reducing Gaussian "floaters" common in free-form 3DGS-SLAM |

By retaining ORB-SLAM2's robust relocalization, GLAM-SLAM achieves state-of-the-art
loop-closure accuracy while maintaining photorealistic rendering.

### Stipple (2026)

**Stereo-inertial VIO + GPU-agnostic 3DGS**.

- Combines stereo cameras and an IMU for tracking (visual-inertial odometry)
- 3DGS mapping runs entirely on the **CPU** (or any GPU), removing the CUDA dependency
  that limits most 3DGS systems to NVIDIA hardware
- **22–33× faster tracking** than SplaTAM on equivalent hardware
- Designed for embedded deployment (drones, mobile robots) where GPU resources are scarce

### LightSplat (2026)

**Hybrid RGB-D SLAM** with a dual-thread architecture:

1. **Sparse tracking thread**: fast frame-to-frame odometry using sparse depth features
2. **Dense Gaussian submap thread**: builds local 3DGS submaps from keyframes, merges them
   on loop closure

- Lightweight by design — targets resource-constrained platforms
- Submap strategy allows each region to be optimised independently, enabling parallel mapping

### FoundationSLAM (AAAI 2026)

**Depth foundation models** (e.g., Depth Anything V2) guide optical-flow estimation
and depth initialisation inside a SLAM loop.

- Monocular: no depth sensor required — the foundation model provides metric-scale priors
- Flow-guided Gaussian densification: new Gaussians are spawned where optical flow
  indicates novel content
- Achieves strong metric accuracy by fusing learned geometric priors with
  multi-view consistency

### VGGT-SLAM 2.0 (RSS 2026)

**Feed-forward reconstruction for SLAM.**

VGGT (Visual Geometry Grounded Transformer) is a large transformer that takes a short
image sequence and directly outputs camera poses, depth, and 3D points — no iterative
optimisation.

- VGGT-SLAM 2.0 wraps this into a SLAM loop with keyframe management and map fusion
- Runs at **3.5 FPS on Jetson Thor** — the first feed-forward SLAM viable on edge hardware
- Trade-off: lower per-frame accuracy than optimisation-based methods, but much faster
  throughput and no convergence failure modes

### Flash-Mono (ICLR 2026)

**Feed-forward Gaussian attributes from monocular video.**

- A learned model predicts per-pixel Gaussian parameters (mean, covariance, colour,
  opacity) directly — **no per-frame optimisation** at test time
- Trained on large-scale video datasets with multi-view supervision
- Extremely fast: a single forward pass produces the full 3DGS map for a keyframe
- Limitation: relies on the training distribution; out-of-domain scenes degrade

### MonoGS++ (Xiao et al., 2025)

**5.57× faster** monocular 3DGS-SLAM via decoupled tracking and mapping:

- **Key insight**: MonoGS couples pose estimation and Gaussian optimisation, creating a
  bottleneck. MonoGS++ decouples them — tracking uses rendered depth/colour comparison,
  mapping runs asynchronously on keyframes
- Back-projected depth initialisation from a monocular depth prior replaces the
  slow per-frame Gaussian densification
- Achieves comparable accuracy to MonoGS at nearly 6× the speed

### GaussianFlow SLAM (Seo et al., RA-L 2026)

Uses **optical flow as geometric supervision** for monocular 3DGS-SLAM:

$$\min_{\theta, T} \sum_{i,j} \| \text{GaussianFlow}_{i \to j}(\theta, T) - \text{OpticalFlow}_{i \to j} \|^2$$

- **GaussianFlow**: the projected 2D motion of each Gaussian between frames, computed
  analytically from the Gaussian parameters and relative camera pose
- Aligning this with estimated optical flow provides dense geometric constraints that
  regularise both tracking and mapping in textureless/ambiguous regions
- Normalised error-based densification prunes unstable Gaussians

### GeoGS-SLAM (Xu et al., 2026)

**Geometry-only** Gaussian SLAM — separates geometry from appearance:

- Uses only depth and normal cues for tracking and mapping (no photometric loss)
- Gaussians represent surface geometry via oriented disks with learned SDF-like properties
- Particularly robust in low-texture environments where photometric methods fail
- Achieves competitive reconstruction quality with ~40% fewer Gaussians

### Taxonomy of 3DGS-SLAM Approaches

| System | Year | Sensor | Tracking | Backend | Real-time? | Key Innovation |
|--------|------|--------|----------|---------|------------|----------------|
| SplaTAM | 2024 | RGB-D | Render + compare | 3DGS optimise | ✓ | First 3DGS-SLAM |
| MonoGS | 2024 | Mono | Photometric | 3DGS + geom. reg. | ✓ | No depth sensor |
| MonoGS++ | 2025 | Mono | Decoupled render | Async 3DGS | ✓ | 5.57× faster than MonoGS |
| MASt3R-SLAM | 2025 | Mono | Foundation model | Pointmap align | ✓ | First GFM SLAM |
| EGG-Fusion | 2026 | RGB-D | Uncertainty | Gaussian surfels | ✓ | 0.6 cm on Replica |
| GLAM-SLAM | 2026 | Mono | ORB features | Anchored 3DGS | ✓ | Anchor grid regularisation |
| GaussianFlow | 2026 | Mono | Optical flow | Flow-regularised 3DGS | ✓ | Dense geometric cues |
| GeoGS-SLAM | 2026 | RGB-D | Depth + normals | Geometry-only 3DGS | ✓ | No photometric loss |
| Stipple | 2026 | Stereo-IMU | VIO | CPU 3DGS | ✓ | GPU-agnostic, 22-33× faster |
| LightSplat | 2026 | RGB-D | Sparse features | Dual-thread submaps | ✓ | Online loop closure via 3DGS registration |
| GPS-SLAM | 2026 | RGB-D | SDF-based align | Gaussian+SDF hybrid | ✓ | **252 FPS on Replica** (50% fewer Gaussians) |
| RTG-SLAM | 2024 | RGB-D | TSDF tracking | Opaque/transparent split | ✓ | 2× speed of NeRF-SLAM at half memory |
| FoundationSLAM | 2026 | Mono | Flow + depth FM | 3DGS densify | ✓ | Foundation depth priors |
| VGGT-SLAM 2.0 | 2026 | Mono | Feed-forward | Transformer | ✓ | 3.5 FPS on Jetson Thor |
| Flash-Mono | 2026 | Mono | Feed-forward | Predicted Gaussians | ✓ | No per-frame optimisation |

### GPS-SLAM: The Speed Frontier (2026)

GPS-SLAM (Gaussian-Plus-SDF) represents the current speed frontier for high-fidelity
RGB-D SLAM, achieving **150+ FPS on real-world sequences** and **252 FPS on Replica**.
The key insight is a **hybrid representation**:

- A colorised **Signed Distance Field (SDF)** captures smooth geometry and base appearance
  via efficient RGB-D fusion (as in classical TSDF methods)
- **3D Gaussians** capture only residual appearance details not modelled by the SDF
- This reduces Gaussian count by ~50% (the SDF handles the bulk of geometry) and
  reduces optimisation iterations by ~75% (Gaussians need only refine appearance)

The architecture separates SLAM into three per-frame stages:
1. SDF-based camera pose alignment (tracking)
2. Depth + RGB integration into the SDF volume
3. Photometric Gaussian optimisation with dynamic add/remove

### Why 3DGS for SLAM Maps?

| Property | Point Cloud | Mesh | NeRF | 3DGS |
|----------|------------|------|------|------|
| Rendering quality | Low | Medium | High | High |
| Rendering speed | Fast | Fast | Slow | **Real-time** |
| Differentiable | No | Partially | Yes | **Yes** |
| Compact | Yes | Yes | Yes | Medium |
| Editable | Yes | Yes | Hard | **Easy** |

**The trend**: tracking and mapping are converging toward **feed-forward prediction** by
large pretrained models, replacing iterative optimisation with amortised inference.

### GSO-SLAM: Bidirectionally Coupled VO + 3DGS (2026)

GSO-SLAM formulates the coupling between visual odometry and Gaussian splatting as an
**Expectation-Maximization** problem:
- **E-step**: VO produces semi-dense depth and camera poses
- **M-step**: Gaussian map is optimised to explain both the depth and RGB observations

The key innovation is **bidirectional coupling** — VO depth estimates are refined by the
Gaussian map (not just consumed), and the map in turn benefits from VO's robust tracking.
This avoids the redundant depth estimation that plagues systems that run VO and 3DGS
independently. Runs at real-time on monocular input.

### ViMGS-SLAM: Transformer-Guided Monocular 3DGS (2026)

ViMGS-SLAM integrates a **Multi-scale Vision Transformer (MViT)** to generate metric depth
priors for 3DGS SLAM:
- MViT produces hierarchical depth at 3 scales and 5 feature levels
- Depth priors initialise Gaussian positions (replacing heuristic SfM initialisation)
- Anisotropic regularisation prevents Gaussians from degenerating into thin "plates"
- Achieves **PSNR 39.6 dB** on Replica (surpassing MonoGS's 35.2) and 46% lower ATE

### Comprehensive Survey Reference

For a complete taxonomy and benchmark tables, see:
- **Tosi et al., "How NeRFs and 3D Gaussian Splatting Are Reshaping SLAM"**, *IEEE T-RO*, 2026 — the first comprehensive survey covering 50+ systems across NeRF-SLAM and 3DGS-SLAM
- **Wang et al., "Towards Next-Generation SLAM: A Survey on 3DGS-SLAM"**, arXiv 2602.04251 — focused analysis across rendering quality, tracking accuracy, speed, and memory

---

## Inverse Rendering (Reference)

NeRF/3DGS learn a radiance field $\mathbf{c}(\mathbf{x}, \mathbf{d})$ that
*bakes* materials and illumination together. **Inverse rendering** decomposes
appearance into geometry, materials (BRDF), and lighting — enabling relighting
and material editing.

### The Rendering Equation (inverse problem)

Given observed $L_o$, recover $f_r$ (BRDF) and $L_i$ (lighting) from:

$$L_o(\mathbf{x}, \omega_o) = L_e + \int_{\mathcal{H}} f_r(\mathbf{x}, \omega_i, \omega_o)\, L_i(\mathbf{x}, \omega_i)\, \cos\theta_i \, d\omega_i$$

### PBR-NeRF (2025)

Jointly estimates SDF geometry + Disney BRDF (albedo $\rho$, roughness $\alpha$,
metalness $m$) + incident light field. The **Cook-Torrance** specular BRDF:

$$f_s = \frac{D(\mathbf{h}; \alpha)\, G(\omega_i, \omega_o; \alpha)\, F(\omega_o, \mathbf{h})}{4\,(\mathbf{n} \cdot \omega_i)(\mathbf{n} \cdot \omega_o)}$$

where $D$ is the GGX normal distribution, $G$ is Smith masking-shadowing,
and $F$ is Schlick's Fresnel approximation. Energy conservation
$\int f_r \cos\theta \, d\omega \leq 1$ prevents highlight baking.

---

## Non-Rigid / Deformable SLAM (Reference)

All SLAM above assumes a **rigid scene**. Non-rigid SLAM breaks this assumption.

### Deformation Model

Each primitive (Gaussian/point) has a canonical position $\mathbf{x}_0$ and a
time-dependent warp field $W(\mathbf{x}_0, t) \to \delta\mathbf{x}$, typically
parameterised by an MLP. The deformed position at time $t$:

$$\mathbf{x}(t) = \mathbf{x}_0 + W_\theta(\mathbf{x}_0, t)$$

with **ARAP** (as-rigid-as-possible) regularisation penalising non-rigid stretch:

$$\mathcal{L}_{\text{ARAP}} = \sum_{(i,j) \in \mathcal{E}} \| (\mathbf{x}_i(t) - \mathbf{x}_j(t)) - R_{ij}(\mathbf{x}_i^0 - \mathbf{x}_j^0) \|^2$$

**4DTAM** (CVPR 2025): 2D Gaussian surface primitives + MLP warp.

**NRGS-SLAM** (2026): each Gaussian gets a learnable deformation probability
via Bayesian self-supervision — prioritise rigid regions for tracking.

---

## Generative 3D — Score Distillation (Reference)

### Score Distillation Sampling (SDS)

**DreamFusion** (2022): distil a 2D diffusion model $\epsilon_\phi$ into a 3D
representation (NeRF/3DGS) parameterised by $\theta$. The SDS gradient:

$$\nabla_\theta \mathcal{L}_{\text{SDS}} = \mathbb{E}_{t,\epsilon}\!\left[w(t)\bigl(\epsilon_\phi(x_t; y, t) - \epsilon\bigr) \frac{\partial x}{\partial \theta}\right]$$

where $x = \text{render}(\theta, c)$, $x_t = \alpha_t x + \sigma_t \epsilon$,
and $y$ is the text prompt. SDS minimises the KL divergence between the
distribution of rendered views and the diffusion prior.

**Janus problem**: each view is optimised independently, producing multi-faced
objects. Mitigated by view-dependent prompting and 3D-aware diffusion.

**GSGEN** (CVPR 2024) replaces NeRF with 3DGS for faster generation.

**PixGS**: ~1s image-to-3DGS via feed-forward Gaussian prediction.

**Connection to mapping**: Dream-SLAM uses generative 3D to hallucinate unseen
structure for exploration planning.

---
## 16. Exercises

### Exercise 1: Volume Rendering Deep Dive

Implement the volume rendering equation and explore its properties.

### Exercise 2: Build Your Own 3DGS Renderer

Create a set of 3D Gaussians and render them from multiple viewpoints.

### Exercise 3: NeRF vs 3DGS Comparison

Compare the two representations on a synthetic scene.

In [ ]:
# ── Exercise 1: Volume Rendering Exploration ──

np.random.seed(42)

print("=" * 60)
print("Exercise 1: Exploring Volume Rendering Properties")
print("=" * 60)

N_S = 128
t_ex = np.linspace(0, 10, N_S)
dt_ex = t_ex[1] - t_ex[0]

# Scene: three Gaussian density bumps at different depths
def make_scene(density_scale=1.0):
    sigma = density_scale * (
        4.0 * np.exp(-((t_ex - 2.5) / 0.3)**2) +
        6.0 * np.exp(-((t_ex - 5.0) / 0.5)**2) +
        3.0 * np.exp(-((t_ex - 8.0) / 0.4)**2)
    )
    colors = np.zeros((N_S, 3))
    colors[:, 0] = np.exp(-((t_ex - 2.5) / 0.6)**2)  # Red near
    colors[:, 1] = np.exp(-((t_ex - 5.0) / 0.8)**2)  # Green middle
    colors[:, 2] = np.exp(-((t_ex - 8.0) / 0.6)**2)  # Blue far
    # Normalize
    total = colors.sum(axis=1, keepdims=True) + 1e-8
    colors = colors / total * 0.9 + 0.05
    return sigma, colors

# (a) Effect of density scaling
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
density_scales = [0.3, 1.0, 5.0]

for idx, ds in enumerate(density_scales):
    sigma, colors = make_scene(ds)
    deltas_ex = np.full(N_S, dt_ex)
    C, weights, T, alphas = volume_render(sigma, colors, deltas_ex)
    
    axes[0, idx].fill_between(t_ex, sigma, alpha=0.3, color='purple')
    axes[0, idx].plot(t_ex, sigma, 'purple', lw=2)
    axes[0, idx].set_xlabel('t (ray parameter)')
    axes[0, idx].set_ylabel('σ(t)')
    axes[0, idx].set_title(f'Density (scale={ds})')
    axes[0, idx].set_ylim(0, max(sigma) * 1.1); axes[0, idx].grid(True)
    
    axes[1, idx].fill_between(t_ex, weights, alpha=0.4, color='orange')
    axes[1, idx].plot(t_ex, weights, 'orange', lw=2)
    axes[1, idx].plot(t_ex, T, 'g--', lw=1.5, label='Transmittance')
    axes[1, idx].set_xlabel('t (ray parameter)')
    axes[1, idx].set_ylabel('Weight / Transmittance')
    axes[1, idx].set_title(f'Weights | Color=({C[0]:.2f},{C[1]:.2f},{C[2]:.2f})')
    axes[1, idx].legend(fontsize=8); axes[1, idx].grid(True)

plt.suptitle('Exercise 1a: Effect of Density Scaling on Volume Rendering', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("Low density → light passes through, all objects contribute.")
print("High density → front objects occlude back ones (near-surface rendering).")

In [ ]:
# (b) Effect of number of samples (quadrature quality)

sample_counts = [8, 16, 32, 64, 128, 512]
ref_sigma, ref_colors = make_scene(1.0)
ref_deltas = np.full(N_S, dt_ex)
C_ref, _, _, _ = volume_render(ref_sigma, ref_colors, ref_deltas)

errors = []
rendered_colors = []

for ns in sample_counts:
    t_sub = np.linspace(0, 10, ns)
    dt_sub = t_sub[1] - t_sub[0]
    sigma_sub = np.interp(t_sub, t_ex, ref_sigma)
    colors_sub = np.column_stack([
        np.interp(t_sub, t_ex, ref_colors[:, c]) for c in range(3)
    ])
    deltas_sub = np.full(ns, dt_sub)
    C_sub, _, _, _ = volume_render(sigma_sub, colors_sub, deltas_sub)
    errors.append(norm(C_sub - C_ref))
    rendered_colors.append(C_sub)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].semilogy(sample_counts, errors, 'bo-', ms=8, lw=2)
axes[0].set_xlabel('Number of Samples')
axes[0].set_ylabel('Error vs Reference (512 samples)')
axes[0].set_title('Quadrature Error vs Number of Ray Samples')
axes[0].grid(True)

for i, (ns, C) in enumerate(zip(sample_counts, rendered_colors)):
    rect = plt.Rectangle((i, 0), 1, 1, color=np.clip(C, 0, 1))
    axes[1].add_patch(rect)
    axes[1].text(i + 0.5, 0.5, f'N={ns}', ha='center', va='center', fontsize=9,
                 color='white' if np.mean(C) < 0.5 else 'black')
axes[1].set_xlim(0, len(sample_counts))
axes[1].set_ylim(0, 1)
axes[1].set_title('Rendered Colors at Different Sample Counts')
axes[1].axis('off')

plt.suptitle('Exercise 1b: Sample Count vs Rendering Quality', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Exercise 2: Custom 3DGS Scene ──

np.random.seed(2025)

print("=" * 60)
print("Exercise 2: Build Your Own 3D Gaussian Splatting Scene")
print("=" * 60)

# Create a more structured scene: a simple "room" with objects
custom_gaussians = []

# Floor
for x in np.linspace(-4, 4, 9):
    for z in np.linspace(3, 11, 9):
        custom_gaussians.append(Gaussian3D(
            mu=[x, 2.0, z], scales=[0.6, 0.01, 0.6],
            quat=[1, 0, 0, 0], color=[0.6, 0.6, 0.55], opacity=0.95
        ))

# Red cube-ish object (cluster of Gaussians)
for dx in [-0.3, 0, 0.3]:
    for dy in [-0.3, 0, 0.3]:
        for dz in [-0.3, 0, 0.3]:
            custom_gaussians.append(Gaussian3D(
                mu=[-2.0 + dx, 0.5 + dy, 6.0 + dz],
                scales=[0.25, 0.25, 0.25],
                quat=[1, 0, 0, 0],
                color=[0.85, 0.15, 0.1],
                opacity=0.9
            ))

# Green cylinder-ish (tall, thin Gaussians)
for dy in np.linspace(-0.8, 0.8, 8):
    q = Rotation.from_euler('z', np.random.uniform(0, np.pi)).as_quat()
    q = np.array([q[3], q[0], q[1], q[2]])
    custom_gaussians.append(Gaussian3D(
        mu=[1.5, 0.2 + dy, 7.0],
        scales=[0.3, 0.15, 0.3],
        quat=q,
        color=[0.15, 0.75, 0.2],
        opacity=0.85
    ))

# Blue sphere
for _ in range(25):
    offset = np.random.randn(3) * 0.3
    q = Rotation.random().as_quat()
    q = np.array([q[3], q[0], q[1], q[2]])
    custom_gaussians.append(Gaussian3D(
        mu=np.array([0.0, -0.3, 5.0]) + offset,
        scales=np.random.uniform(0.1, 0.3, 3),
        quat=q,
        color=[0.1, 0.3, 0.9],
        opacity=0.8
    ))

print(f"Custom scene: {len(custom_gaussians)} Gaussians")

# Render from multiple viewpoints
K_ex = np.array([[100, 0, 56], [0, 100, 40], [0, 0, 1]], dtype=np.float64)
n_views = 4
view_angles = np.linspace(-0.4, 0.4, n_views)

fig, axes = plt.subplots(1, n_views, figsize=(16, 4))

for idx, angle in enumerate(view_angles):
    T = make_camera_pose(angle, distance=2.0, height=0.0)
    img = render_gaussians(custom_gaussians, T, K_ex, img_h=80, img_w=112)
    axes[idx].imshow(img)
    axes[idx].set_title(f'θ = {np.degrees(angle):.0f}°')
    axes[idx].axis('off')

plt.suptitle('Exercise 2: Custom 3DGS Scene — Multi-View Rendering', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Exercise 3: NeRF vs 3DGS Comparison ──

print("=" * 60)
print("Exercise 3: NeRF vs 3DGS — Conceptual Comparison")
print("=" * 60)

# Render the same scene with both methods and compare

# (a) NeRF-style: ray marching through Gaussian density field
IMG_CMP_H, IMG_CMP_W = 48, 64
FOCAL_CMP = 50.0
N_SAMPLES_CMP = 48
t_near_c, t_far_c = 2.0, 12.0
t_cmp = np.linspace(t_near_c, t_far_c, N_SAMPLES_CMP)
dt_cmp = (t_far_c - t_near_c) / N_SAMPLES_CMP

# Simple scene: two colored blobs (same for both renderers)
blobs = [
    {'center': np.array([-1.0, 0.0, 5.0]), 'radius': 0.8, 'color': np.array([0.9, 0.2, 0.1]), 'density': 12.0},
    {'center': np.array([1.0, 0.3, 7.0]), 'radius': 1.0, 'color': np.array([0.1, 0.3, 0.9]), 'density': 10.0},
    {'center': np.array([0.0, -0.5, 4.5]), 'radius': 0.6, 'color': np.array([0.9, 0.8, 0.1]), 'density': 15.0},
]

def blob_density_color(pts):
    sigma = np.zeros(len(pts))
    color = np.ones((len(pts), 3)) * 0.95
    for b in blobs:
        d = norm(pts - b['center'], axis=1)
        s = b['density'] * np.exp(-0.5 * (d / (b['radius'] * 0.5))**2)
        w = s / (sigma + s + 1e-10)
        color = w[:, None] * b['color'] + (1 - w[:, None]) * color
        sigma += s
    return sigma, color

import time

# NeRF rendering
t0 = time.time()
img_nerf = np.zeros((IMG_CMP_H, IMG_CMP_W, 3))
for v in range(IMG_CMP_H):
    for u in range(IMG_CMP_W):
        d = np.array([(u - IMG_CMP_W/2) / FOCAL_CMP, (v - IMG_CMP_H/2) / FOCAL_CMP, 1.0])
        d /= norm(d)
        pts = t_cmp[:, None] * d
        sigmas, colors = blob_density_color(pts)
        C, _, _, _ = volume_render(sigmas, colors, np.full(N_SAMPLES_CMP, dt_cmp))
        img_nerf[v, u] = np.clip(C, 0, 1)
t_nerf = time.time() - t0

# 3DGS rendering: represent blobs as Gaussians
gs_blobs = []
for b in blobs:
    for _ in range(15):
        offset = np.random.randn(3) * b['radius'] * 0.3
        q = Rotation.random().as_quat()
        gs_blobs.append(Gaussian3D(
            mu=b['center'] + offset,
            scales=np.random.uniform(0.15, 0.4, 3) * b['radius'],
            quat=np.array([q[3], q[0], q[1], q[2]]),
            color=b['color'],
            opacity=0.7
        ))

K_cmp = np.array([[FOCAL_CMP, 0, IMG_CMP_W/2],
                   [0, FOCAL_CMP, IMG_CMP_H/2],
                   [0, 0, 1]], dtype=np.float64)

t0 = time.time()
img_3dgs = render_gaussians(gs_blobs, np.eye(4), K_cmp, IMG_CMP_H, IMG_CMP_W)
t_3dgs = time.time() - t0

# Compare
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_nerf)
axes[0].set_title(f'NeRF-style (ray marching)\nTime: {t_nerf:.2f}s')
axes[0].axis('off')

axes[1].imshow(img_3dgs)
axes[1].set_title(f'3DGS (splatting)\nTime: {t_3dgs:.2f}s')
axes[1].axis('off')

# Comparison table
comparison_text = (
    "NeRF vs 3DGS Comparison\n"
    "━━━━━━━━━━━━━━━━━━━━━━\n"
    f"NeRF render time:  {t_nerf:.2f}s\n"
    f"3DGS render time:  {t_3dgs:.2f}s\n\n"
    "Property      NeRF    3DGS\n"
    "──────────────────────────\n"
    "Repr.     Implicit Explicit\n"
    "Render    Ray-march Splat\n"
    "GPU RT    Slow     Fast\n"
    "Edit      Hard     Easy\n"
    "Geometry  Fuzzy    Ellipsoidal\n"
    "Storage   Compact  Medium\n"
)
axes[2].text(0.05, 0.95, comparison_text, transform=axes[2].transAxes,
             fontsize=10, va='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[2].axis('off')
axes[2].set_title('Comparison')

plt.suptitle('Exercise 3: NeRF vs 3D Gaussian Splatting', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nNote: Both are pure Python here. Real 3DGS on GPU is 100-1000x faster than NeRF.")
print(f"3DGS achieves real-time (>100 FPS) at 1080p; NeRF typically ~0.1-1 FPS.")

---

## Summary

| Topic | Key Concept |
|-------|-------------|
| **Volume Rendering** | $C = \int T(t) \sigma(t) c(t) dt$ — accumulate color weighted by transmittance |
| **NeRF** | MLP maps $(x,y,z,\theta,\phi) \to (\sigma, c)$, trained via photometric loss |
| **3D Gaussian Splatting** | Explicit Gaussians $(\mu, \Sigma, c, \alpha)$, EWA splatting, real-time rendering |
| **Differentiable Rendering** | Analytic gradients $\partial C / \partial \mu, \partial C / \partial \Sigma$ enable optimization |
| **3DGS-SLAM** | SplaTAM, MonoGS, EGG-Fusion — Gaussians as the SLAM map |
| **MASt3R-SLAM** | Foundation model predicts 3D pointmaps directly |
| **Neural Surfaces** | NeuS, VolSDF — SDF-based for clean geometry |
| **Inverse Rendering** | Decompose appearance into materials + lighting |
| **Non-Rigid SLAM** | Handle deformable scenes with canonical + deformation model |
| **Generative 3D** | SDS loss, text/image → 3D via diffusion models |

**The field is moving extremely fast**. As of 2026, 3DGS-based methods dominate for
real-time applications, while foundation models (MASt3R) are reshaping the SLAM pipeline.

### Key Takeaways

- **Volume rendering is the unifying equation**: both NeRF and 3DGS render by accumulating colour weighted by transmittance along rays — the difference is implicit (MLP) vs explicit (Gaussian) scene representation
- **3D Gaussian Splatting achieves real-time rendering** (100+ FPS) by replacing per-ray MLP queries with rasterisation of sorted 2D Gaussian splats — a paradigm shift for practical deployment
- **Differentiable rendering enables optimisation**: analytic gradients through the rendering equation let us optimise scene parameters (positions, colours, opacities) from 2D image supervision alone
- **3DGS-SLAM is the frontier**: systems like SplaTAM and MonoGS use Gaussians as the SLAM map, enabling simultaneous tracking, mapping, and photorealistic novel-view synthesis
- **Foundation models (MASt3R, VGGT) skip the pipeline**: feed-forward transformers predict 3D pointmaps directly from image pairs, avoiding feature extraction, matching, and triangulation entirely
- **Neural implicit surfaces (NeuS, VolSDF) trade speed for geometry quality**: replacing density with signed distance functions yields clean, watertight meshes at the cost of slower convergence
- **Inverse rendering decomposes appearance**: separating materials, lighting, and geometry enables relighting, material editing, and physics-based simulation from captured scenes